## Load the dataset




In [1]:
import os
import cv2
import numpy as np

dataset_path = "/content/drive/MyDrive/Prodigy/leapGestRecog"
images = []
labels = []

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_path):
        for image_name in os.listdir(class_path):
            image_path = os.path.join(class_path, image_name)
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE) # Read as grayscale
            if img is not None:
                images.append(img)
                labels.append(class_name)

images = np.array(images)
labels = np.array(labels)

print("Shape of images array:", images.shape)
print("Shape of labels array:", labels.shape)

Shape of images array: (0,)
Shape of labels array: (0,)


In [2]:
import os
import cv2
import numpy as np

dataset_path = "/content/drive/MyDrive/Prodigy/leapGestRecog"
images = []
labels = []

# Check the directory structure
print("Contents of dataset_path:", os.listdir(dataset_path))

# Assuming there are subdirectories within each class directory for each person
for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_path):
        print(f"Processing class: {class_name}")
        for person_folder in os.listdir(class_path):
            person_path = os.path.join(class_path, person_folder)
            if os.path.isdir(person_path):
                print(f"Processing person folder: {person_folder}")
                for image_name in os.listdir(person_path):
                    image_path = os.path.join(person_path, image_name)
                    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE) # Read as grayscale
                    if img is not None:
                        images.append(img)
                        labels.append(class_name)

images = np.array(images)
labels = np.array(labels)

print("Shape of images array:", images.shape)
print("Shape of labels array:", labels.shape)

Contents of dataset_path: ['01', '02', '00', '04', '08', '09', '03', '05', '06', '07', 'leapGestRecog']
Processing class: 01
Processing person folder: 01_palm
Processing person folder: 02_l
Processing person folder: 03_fist
Processing person folder: 05_thumb
Processing person folder: 04_fist_moved
Processing person folder: 06_index
Processing person folder: 07_ok
Processing person folder: 08_palm_moved
Processing person folder: 09_c
Processing person folder: 10_down
Processing class: 02
Processing person folder: 01_palm
Processing person folder: 02_l
Processing person folder: 03_fist
Processing person folder: 04_fist_moved
Processing person folder: 05_thumb
Processing person folder: 06_index
Processing person folder: 07_ok
Processing person folder: 08_palm_moved
Processing person folder: 09_c
Processing person folder: 10_down
Processing class: 00
Processing person folder: 01_palm
Processing person folder: 02_l
Processing person folder: 03_fist
Processing person folder: 04_fist_moved
Pr

## Preprocess the data




In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Resize images
resized_images = []
target_size = (64, 64)
for img in images:
    resized_img = cv2.resize(img, target_size)
    resized_images.append(resized_img)
resized_images = np.array(resized_images)

# Normalize pixel values
normalized_images = resized_images.astype('float32') / 255.0

# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    normalized_images, encoded_labels, test_size=0.2, random_state=42
)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (16000, 64, 64)
Shape of X_test: (4000, 64, 64)
Shape of y_train: (16000,)
Shape of y_test: (4000,)


## Build the model




In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential()

# Add convolutional layers
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(target_size[0], target_size[1], 1))) # Assuming grayscale images (1 channel)
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Flatten the output
model.add(Flatten())

# Add dense layers
model.add(Dense(128, activation='relu'))

# Add output layer
num_classes = len(np.unique(encoded_labels)) # Get the number of unique classes
model.add(Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,914 (2.61 MB)

 Trainable params: 683,914 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

## Train the model




In [5]:
epochs = 10
batch_size = 32

history = model.fit(X_train, y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_data=(X_test, y_test))

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 112s 218ms/step - accuracy: 0.6812 - loss: 0.9218 - val_accuracy: 0.9902 - val_loss: 0.0365
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 105s 210ms/step - accuracy: 0.9901 - loss: 0.0367 - val_accuracy: 0.9900 - val_loss: 0.0203
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 150s 227ms/step - accuracy: 0.9904 - loss: 0.0205 - val_accuracy: 0.9910 - val_loss: 0.0186
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 139s 221ms/step - accuracy: 0.9851 - loss: 0.0397 - val_accuracy: 0.9915 - val_loss: 0.0152
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 142s 222ms/step - accuracy: 0.9915 - loss: 0.0158 - val_accuracy: 0.9893 - val_loss: 0.0243
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 147s 232ms/step - accuracy: 0.9930 - loss: 0.0124 - val_accuracy: 0.9910 - val_loss: 0.0141
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 112s 224ms/step - accuracy: 0.9922 - loss: 0.0132 - val_accuracy: 0.9915 - val_loss: 0.0145
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 142s 223ms/step - accuracy: 0.9924 -

In [11]:
df = pd.read_csv('sample_data/california_housing_train.csv')
display(df.head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


## Evaluate the model



In [12]:
loss, accuracy = model.evaluate(X_test, y_test)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.9905 - loss: 0.0172
Test Loss: 0.0162
Test Accuracy: 0.9912
